Let's first load, inspect and clean the data

In [4]:
import glob, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

DATA_DIR = "sample_data/sample_data_part_3"  

In [24]:
## check if we got all the files, should have 2 per year
files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv*")))
print(len(files), "files found:")
for f in files:
    print(" ", os.path.basename(f))

16 files found:
  mpi_saale_2009a.csv
  mpi_saale_2009b.csv
  mpi_saale_2010a.csv
  mpi_saale_2010b.csv
  mpi_saale_2011a.csv
  mpi_saale_2011b.csv
  mpi_saale_2012a.csv
  mpi_saale_2012b.csv
  mpi_saale_2013a.csv
  mpi_saale_2013b.csv
  mpi_saale_2014a.csv
  mpi_saale_2014b.csv
  mpi_saale_2015a.csv
  mpi_saale_2015b.csv
  mpi_saale_2016a.csv
  mpi_saale_2016b.csv


In [25]:
# look at one of the files to see its shape and columns
sample = pd.read_csv(files[0])
print(sample.shape)
print(sample.columns.tolist())
sample.head()

(26064, 31)
['Date Time', 'p (mbar)', 'T (degC)', 'rh (%)', 'sh (g/kg)', 'Tpot (K)', 'Tdew (degC)', 'VPmax (mbar)', 'VPact (mbar)', 'VPdef (mbar)', 'H2OC (mmol/mol)', 'rho (g/m**3)', 'wv (m/s)', 'wd (deg)', 'rain (mm)', 'SWDR (W/m**2)', 'SDUR (s)', 'TRAD (degC)', 'Rn (W/m**2)', 'ST002 (degC)', 'ST004 (degC)', 'ST008 (degC)', 'ST016 (degC)', 'ST032 (degC)', 'ST064 (degC)', 'ST128 (degC)', 'SM008 (%)', 'SM016 (%)', 'SM032 (%)', 'SM064 (%)', 'SM128 (%)']


,Date Time,p (mbar),T (degC),rh (%),sh (g/kg),Tpot (K),Tdew (degC),VPmax (mbar),VPact (mbar),VPdef (mbar),...,ST008 (degC),ST016 (degC),ST032 (degC),ST064 (degC),ST128 (degC),SM008 (%),SM016 (%),SM032 (%),SM064 (%),SM128 (%)
0,01.01.2009 00:10:04,1007.35,-9.45,94.8,1.75,263.16,-10.12,2.98,2.82,0.15,...,-0.51,0.72,2.26,4.74,7.63,17.88,30.74,34.3,33.57,33.75
1,01.01.2009 00:20:04,1007.40,-9.52,94.8,1.74,263.08,-10.19,2.96,2.81,0.15,...,-0.52,0.72,2.25,4.74,7.63,17.86,30.74,34.3,33.57,33.75
2,01.01.2009 00:30:04,1007.39,-9.61,94.7,1.72,263.00,-10.29,2.94,2.78,0.16,...,-0.53,0.71,2.25,4.73,7.63,17.83,30.74,34.3,33.57,33.75
3,01.01.2009 00:40:04,1007.35,-9.86,94.0,1.67,262.75,-10.64,2.88,2.71,0.17,...,-0.53,0.71,2.25,4.73,7.63,17.80,30.69,34.3,33.57,33.75
4,01.01.2009 00:50:04,1007.36,-9.76,94.6,1.70,262.85,-10.46,2.91,2.75,0.16,...,-0.54,0.70,2.25,4.73,7.63,17.77,30.69,34.3,33.57,33.75


In [26]:
# fuse the code tghter

# parse thru all the files and append
dfs = []
for f in files:
    part = pd.read_csv(f)
    part["source_file"] = os.path.basename(f)   # to debugging
    dfs.append(part)

# fuse all data tgether
df = pd.concat(dfs, ignore_index=True)

# get date time column into datatime format. clean up errors
df["Date Time"] = pd.to_datetime(df["Date Time"], format="%d.%m.%Y %H:%M:%S", errors="coerce")

# sort chronologically
df = df.sort_values("Date Time").reset_index(drop=True)

# Remove duplicates, can occur at start and end of file
n_before = len(df)
df = df.drop_duplicates(subset="Date Time", keep="first").reset_index(drop=True)
print(f"Dropped {n_before - len(df)} duplicate timestamps")

print("Shape:", df.shape)
print("Date range:", df["Date Time"].min(), "->", df["Date Time"].max())
print("Unparsed dates:", df["Date Time"].isna().sum())

Dropped 38 duplicate timestamps
Shape: (419477, 32)
Date range: 2009-01-01 00:10:04 -> 2017-01-01 00:00:04
Unparsed dates: 0


In [ ]:
# df = df.drop(columns=["source_file"])

If the prof has given the proper dataset, we can ignore this followig code

In [27]:
# since we want hourly gaps, we clean the gaps for the extra timestamps
# round to minutes
df["Date Time"] = df["Date Time"].dt.round("min")

# keep one data per hour
df = df[df["Date Time"].dt.minute == 0].reset_index(drop=True)

# drop duplicated created from rounding per hour
df = df.drop_duplicates(subset="Date Time", keep="first").reset_index(drop=True)

print("Hourly shape:", df.shape)   # expect roughly 70,000 rows

Hourly shape: (69911, 32)


ltsm requires that its data tiem are linear in time. That means if there are some gaps in irregular time stamps, it could cause issues. The model odesnt necessarily know that more time has passed between the two points.

One way to solve it is to interpolate a value, but assignemtn removes points if there's data leakage or such things.

We can thus drop some sliding windows over these areas. Since there are not that many gaps we are not losing much traning data.

In [28]:
dt = df["Date Time"].diff().dropna()
print("Time step counts:")
print(dt.value_counts().head())

# Locate any gaps (anything other than 1 hour)
gaps = df.loc[dt[dt != pd.Timedelta(hours=1)].index, ["Date Time", "source_file"]]
gaps["gap_size"] = dt[dt != pd.Timedelta(hours=1)]
print("\nGaps found:", len(gaps))
print(gaps.head(20))

Time step counts:
Date Time
0 days 01:00:00    69895
0 days 02:00:00        3
0 days 20:00:00        2
0 days 08:00:00        2
2 days 17:00:00        1
Name: count, dtype: int64

Gaps found: 15
                Date Time          source_file        gap_size
17784 2011-01-14 17:00:00  mpi_saale_2011a.csv 2 days 17:00:00
18043 2011-01-25 17:00:00  mpi_saale_2011a.csv 0 days 06:00:00
18897 2011-03-03 17:00:00  mpi_saale_2011a.csv 1 days 11:00:00
23843 2011-09-26 14:00:00  mpi_saale_2011b.csv 0 days 20:00:00
26865 2012-01-30 14:00:00  mpi_saale_2012a.csv 0 days 03:00:00
26914 2012-02-01 16:00:00  mpi_saale_2012a.csv 0 days 02:00:00
27268 2012-02-16 11:00:00  mpi_saale_2012a.csv 0 days 02:00:00
38659 2013-06-05 15:00:00  mpi_saale_2013a.csv 0 days 14:00:00
38767 2013-06-11 09:00:00  mpi_saale_2013a.csv 1 days 07:00:00
39680 2013-07-19 11:00:00  mpi_saale_2013b.csv 0 days 02:00:00
39828 2013-07-26 10:00:00  mpi_saale_2013b.csv 0 days 20:00:00
40138 2013-08-08 15:00:00  mpi_saale_2013b.csv 0 

for each gap that we drop, we add a segment ID (a segment consists of a bunch of non interrupted data)

In [29]:
# new segment id every time the time step isnt 1 hours
df["segment"] = (df["Date Time"].diff() != pd.Timedelta(hours=1)).cumsum()
print("Number of continuous segments:", df["segment"].nunique())
print(df.groupby("segment").size().describe())

Number of continuous segments: 16
count       16.000000
mean      4369.437500
std       6379.066337
min         49.000000
25%        176.250000
50%        604.000000
75%       6557.250000
max      17784.000000
dtype: float64


In [30]:
def make_windows(data, feature_cols, target_cols, k):
    """
    data: DataFrame with a 'segment' column, sorted by time.
    Returns X of shape (N, k, n_features) and y of shape (N, n_targets).
    No window crosses a gap, because each is built inside one segment.
    """
    X_list, y_list = [], []
    for _, seg in data.groupby("segment"):
        if len(seg) <= k:          # segment too short to form even one window
            continue
        feats = seg[feature_cols].to_numpy(dtype=np.float32)
        targs = seg[target_cols].to_numpy(dtype=np.float32)
        for i in range(k, len(seg)):
            X_list.append(feats[i - k:i])   # rows t-k ... t-1
            y_list.append(targs[i])          # row t
    return np.stack(X_list), np.stack(y_list)